OpenAI 함수 호출 기능 활용 AI Agent 구현

In [2]:
import json
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI

# ==========================================
# [1. API 환경 설정]
# -.env 파일로부터 API Key 환경 변수 로드
# ==========================================
load_dotenv()


# ==========================================
# [2. 함수 정의]
# - 비즈니스 로직 함수 정의 및 각 함수 내 예외 처리 구현
# ==========================================

def convert_date_format(date_str, current_format, target_format):
    """날짜 문자열의 포맷을 원하는 형태로 변환하는 함수"""
    try:
        dt = datetime.strptime(date_str, current_format)
        return dt.strftime(target_format)
    except Exception as e:
        return f"날짜 변환 에러 발생: {e}"


def add_numbers(x, y):
    """두 숫자를 입력받아 더한 결과를 문자열로 반환하는 함수"""
    try:
        return str(x + y)
    except Exception as e:
        return f"연산 에러 발생: {e}"


# ==========================================
# [2. 함수 정의 - Function Calling용 JSON schema 정의]
# - 모델이 인식할 수 있도록 함수의 스펙을 OpenAI 규격에 맞춰 정의
# ==========================================
tools = [
    {
        "type": "function",
        "function": {
            "name": "convert_date_format",
            "description": "날짜 문자열의 형식을 원하는 포맷으로 변환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date_str": {
                        "type": "string",
                        "description": "변환할 날짜 문자열 (예: '2026-07-07')",
                    },
                    "current_format": {
                        "type": "string",
                        "description": "현재 날짜 문자열의 포맷 (예: '%Y-%m-%d')",
                    },
                    "target_format": {
                        "type": "string",
                        "description": "변경하고자 하는 목표 포맷 (예: '%Y년 %m월 %d일')",
                    },
                },
                "required": ["date_str", "current_format", "target_format"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "두 개의 숫자를 입력받아 더한 결과를 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "x": {"type": "number", "description": "첫 번째 숫자"},
                    "y": {"type": "number", "description": "두 번째 숫자"},
                },
                "required": ["x", "y"],
            },
        },
    },
]


# ==========================================
# [3. 에이전트 클래스 구현]
# - OpenAI API 호출, 대화 관리, 함수 호출 처리를 전담하는 에이전트 클래스
# ==========================================
class OpenAIAgent:

    def __init__(self):
        # OpenAI 클라이언트 인스턴스 초기화 (환경 변수의 OPENAI_API_KEY 자동 로드)
        self.client = OpenAI()

    def call_openai(self, messages, tools=None, tool_choice="auto"):
        """서브 메서드: OpenAI Chat Completions API를 호출하는 핵심 메서드"""
        return self.client.chat.completions.create(
            model="gpt-4o",
            messages=messages, 
            tools=tools, 
            tool_choice=tool_choice
        )

    def chat(self, user_input, messages, tools):
        """메인 메서드: 사용자 입력을 받아 대화를 처리하고 함수 호출 흐름을 제어"""
        # 1. 사용자의 질문을 대화 기록(messages)에 추가
        messages.append({"role": "user", "content": user_input})

        # [4. 함수 호출 흐름 처리를 위한 루프 구조]
        while True:
            # 2. 모델에게 대화 히스토리와 도구 목록을 전달하여 응답 요청
            response = self.call_openai(messages, tools=tools)
            reply = response.choices[0].message
            # print(reply)

            # 3. 모델의 응답(Assistant 역할)을 대화 기록에 누적 (tool_calls가 포함되어 있을 수 있음)
            messages.append(reply)

            # 4. 모델의 응답에 function_call(tool_calls) 요구가 존재하는지 확인
            if reply.tool_calls:
                # 병렬 함수 호출 기능(Parallel Tool Calling)을 지원하기 위해 반복문 처리
                for tool_call in reply.tool_calls:
                    self.handle_function_call(tool_call, messages)
                # 함수 실행 결과가 messages에 추가되었으므로, 다시 루프 상단으로 이동하여 모델에게 최종 응답 요구
                continue
            else:
                # 더 이상 호출할 함수가 없다면 모델의 최종 자연어 응답 내용을 반환
                return reply.content

    def handle_function_call(self, tool_call, messages):
        """서브 메서드: 모델이 요청한 함수를 실제 매핑 및 실행하고 결과를 기록"""
        function_name = tool_call.function.name
        # print(function_name)
        # convert_date_format
        # add_numbers

        # 모델이 생성한 인자(arguments)를 안전하게 JSON 파싱 처리
        args = json.loads(tool_call.function.arguments)
        # print(args) 
        # {'date_str': '2024-12-25', 'current_format': '%Y-%m-%d', 'target_format': '%Y년 %m월 %d일'}
        # {'x': 23.5, 'y': 3.1}   

        # 함수 이름 비교를 통해 비즈니스 로직 매핑 및 실행 (**args 딕셔너리 언패킹 활용)
        if "add_numbers" in function_name:
            result = add_numbers(**args)
        elif "convert_date_format" in function_name:
            result = convert_date_format(**args)
        else:
            result = f"Error: {function_name} 함수를 찾을 수 없습니다."

        # OpenAI 규격에 맞게 'role': 'tool' 형태의 결과 메시지 구성
        function_messages = {
            "role": "tool",
            "tool_call_id": tool_call.id,  # 어떤 요청에 대한 결과인지 ID 매칭 필수
            "name": function_name,
            "content": result,
        }
        # print(function_messages)
        # {'role': 'tool', 'tool_call_id': 'call_F3HJZBqR7kHpO33e3TQYl95j', 'name': 'add_numbers', 'content': '26.6'}

        # 함수 실행 결과를 대화 기록에 추가
        messages.append(function_messages)

# ==========================================
# [5. 테스트]
# ==========================================
if __name__ == "__main__":
    # 에이전트 인스턴스 생성 및 멀티 턴 기억 장치(대화 기록) 준비
    agent = OpenAIAgent()
    conversation_history = []

    print("--- 테스트 1: 날짜 변환 ---")
    answer1 = agent.chat(
        "2026-12-18을 2026년 12월 18일 형식으로 바꿔줘",
        conversation_history,
        tools,
    )
    print(f"AI 응답: {answer1}\n")

    print("--- 테스트 2: 덧셈 연산 ---")
    answer2 = agent.chat(
        "1000이랑 301.1을 더하면 얼마야?", conversation_history, tools
    )
    print(f"AI 응답: {answer2}\n")

--- 테스트 1: 날짜 변환 ---
AI 응답: 2026년 12월 18일로 변환되었습니다.

--- 테스트 2: 덧셈 연산 ---
AI 응답: 1000과 301.1을 더하면 1301.1입니다.



In [1]:
!pip install openai

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 23.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 38.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
